# 기존 뉴스 URL 데이터 추출

`source_data/*.csv`의 모든 URL에서 제목, 주제, 요약, 본문을 추출해 `model/output` 아래에 카테고리별 CSV로 저장합니다. 임베딩 코드는 포함하지 않습니다.

요약은 언론사가 제공한 JSON-LD/OpenGraph 설명을 우선 사용하고, 없으면 본문의 첫 완결 문장 2~3개를 사용합니다. 생성형 요약이 아니므로 원문에 없는 내용을 만들지 않습니다. `summary_source`에 추출 출처를 남깁니다. 주제는 기사 섹션, 키워드, 제목 순으로 선택합니다.

In [ ]:
%pip install -q pandas requests beautifulsoup4 lxml trafilatura tqdm

In [ ]:
from __future__ import annotations
import hashlib, json, re, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urlsplit
import pandas as pd
import requests, trafilatura
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.casefold() == 'model' else cwd
SOURCE_DIR = ROOT / 'source_data'
CACHE_DIR = ROOT / 'model' / 'article_cache'
OUTPUT_DIR = ROOT / 'model' / 'output'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CRAWL_LIMIT: int | None = None  # 모든 고유 URL 수집
REQUEST_DELAY_SECONDS = 0.25
REQUEST_TIMEOUT_SECONDS = 20
MAX_BODY_CHARS = 50_000
SUMMARY_MAX_CHARS = 500
RETRY_FAILED_CACHE = True
print('프로젝트:', ROOT)
print('카테고리별 출력 폴더:', OUTPUT_DIR)

## 1. 기존 CSV 통합

In [ ]:
CATEGORY_MAP = {
    'Innodep': '이노뎁 관련 기사', 'Security': '보안 관련 기사',
    'Industry Trends': '업계 동향 기사', 'Government/Public': '정부/공공 기사',
    'Venture/Finance': '벤처/금융 기사', 'Production/Wages': '생산/임금 기사',
}
files = sorted(SOURCE_DIR.glob('*.csv'))
if not files: raise FileNotFoundError(SOURCE_DIR)
frames = []
for path in files:
    df = pd.read_csv(path, dtype=str)
    missing = {'date', 'category', 'article_link'} - set(df.columns)
    if missing: raise ValueError(f'{path.name}: 필수 컬럼 누락 {missing}')
    df['source_file'] = path.name
    frames.append(df)
legacy = pd.concat(frames, ignore_index=True)
legacy['digest_date'] = pd.to_datetime(legacy['date'], errors='raise').dt.date.astype(str)
legacy['legacy_category'] = legacy['category'].map(CATEGORY_MAP)
if legacy['legacy_category'].isna().any(): raise ValueError('알 수 없는 카테고리가 있습니다.')
legacy['article_link'] = legacy['article_link'].str.strip()
legacy['article_id'] = legacy['article_link'].map(lambda x: hashlib.sha256(x.encode()).hexdigest()[:24])
articles = legacy.drop_duplicates('article_link')[['article_id', 'article_link']].reset_index(drop=True)
display(legacy.head())
print(f'원본 {len(legacy):,}행, 고유 URL {len(articles):,}개')

## 2. 제목·주제·요약·본문 추출 함수

In [ ]:
def clean(value: Any) -> str:
    if value is None: return ''
    return re.sub(r'\s+', ' ', BeautifulSoup(str(value), 'html.parser').get_text(' ', strip=True)).strip()

def walk_json(value: Any):
    if isinstance(value, dict):
        yield value
        for child in value.values(): yield from walk_json(child)
    elif isinstance(value, list):
        for child in value: yield from walk_json(child)

def keyword_text(value: Any) -> str:
    items = value if isinstance(value, list) else re.split(r'[,|]', clean(value))
    result = []
    for item in items:
        item = clean(item)
        if item and item.casefold() not in [x.casefold() for x in result]: result.append(item)
    return ', '.join(result[:10])

def json_ld_article(soup: BeautifulSoup) -> dict[str, str]:
    best = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        try: payload = json.loads(tag.string or tag.get_text())
        except Exception: continue
        for node in walk_json(payload):
            kinds = node.get('@type', [])
            kinds = kinds if isinstance(kinds, list) else [kinds]
            if not any(str(x).casefold() in {'newsarticle','article','reportagenewsarticle'} for x in kinds): continue
            pub = node.get('publisher', '')
            if isinstance(pub, dict): pub = pub.get('name', '')
            candidate = {
                'title': clean(node.get('headline') or node.get('name')),
                'description': clean(node.get('description')),
                'body': clean(node.get('articleBody')),
                'section': clean(node.get('articleSection')),
                'keywords': keyword_text(node.get('keywords')),
                'published_at': clean(node.get('datePublished')), 'publisher': clean(pub),
            }
            if len(candidate['body']) + 10*bool(candidate['title']) > len(best.get('body','')) + 10*bool(best.get('title')): best = candidate
    return best

def meta(soup: BeautifulSoup, *selectors: str) -> str:
    for selector in selectors:
        tag = soup.select_one(selector)
        if tag and clean(tag.get('content')): return clean(tag.get('content'))
    return ''

def html_body(soup: BeautifulSoup) -> str:
    for tag in soup.select('script,style,nav,header,footer,aside,form'): tag.decompose()
    selectors = ['article','#articleBody','#article-body','#newsEndContents','.article_body','.article-body','.news_body','.view_body','.article_txt']
    candidates = [clean(n.get_text(' ', strip=True)) for s in selectors for n in soup.select(s)]
    candidates = [x for x in candidates if x]
    if candidates: return max(candidates, key=len)
    return ' '.join(clean(p.get_text(' ', strip=True)) for p in soup.select('p') if len(clean(p.get_text())) >= 30)

def lead_summary(body: str, max_sentences: int = 3) -> str:
    sentences = re.split(r'(?<=[.!?。！？])\s+', clean(body))
    selected = []
    for sentence in sentences:
        if len(sentence) < 20: continue
        if len(' '.join(selected + [sentence])) > SUMMARY_MAX_CHARS:
            if not selected: selected = [sentence[:SUMMARY_MAX_CHARS].rstrip() + '…']
            break
        selected.append(sentence)
        if len(selected) == max_sentences: break
    return ' '.join(selected)

def select_summary(tf_desc: str, ld_desc: str, meta_desc: str, body: str):
    for source, value in [('trafilatura_description',tf_desc),('json_ld_description',ld_desc),('meta_description',meta_desc)]:
        value = clean(value)
        if len(value) >= 30: return value[:SUMMARY_MAX_CHARS], source
    value = lead_summary(body)
    return value, ('lead_sentences' if value else 'missing')

## 3. URL 수집 실행

기사별 JSON 캐시를 사용하므로 중단 후 다시 실행할 수 있습니다.

In [ ]:
retry = Retry(total=3, backoff_factor=.8, status_forcelist=(429,500,502,503,504), allowed_methods=frozenset({'GET'}))
session = requests.Session()
session.mount('http://', HTTPAdapter(max_retries=retry)); session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers.update({'User-Agent':'Mozilla/5.0 (compatible; NewsDigestResearch/1.0)','Accept-Language':'ko-KR,ko;q=0.9'})

def extract_url(url: str, article_id: str) -> dict[str, Any]:
    response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS); response.raise_for_status()
    response.encoding = response.apparent_encoding or response.encoding
    html = response.text; soup = BeautifulSoup(html, 'lxml'); ld = json_ld_article(soup)
    tf = {}
    try:
        raw = trafilatura.extract(html, url=response.url, output_format='json', with_metadata=True, include_comments=False, include_tables=False, favor_precision=True)
        if raw: tf = json.loads(raw)
    except Exception: pass
    title = clean(tf.get('title') or ld.get('title') or meta(soup,'meta[property="og:title"]') or (soup.title.get_text() if soup.title else ''))
    body = clean(tf.get('text') or ld.get('body') or html_body(soup))[:MAX_BODY_CHARS]
    section = clean(ld.get('section') or meta(soup,'meta[property="article:section"]'))
    keywords = keyword_text(ld.get('keywords') or meta(soup,'meta[name="keywords"]','meta[property="article:tag"]'))
    topic, topic_source = (section,'article_section') if section else ((keywords,'keywords') if keywords else (title,'title_fallback'))
    summary, summary_source = select_summary(tf.get('description',''), ld.get('description',''), meta(soup,'meta[property="og:description"]','meta[name="description"]'), body)
    publisher = clean(ld.get('publisher') or meta(soup,'meta[property="og:site_name"]') or urlsplit(response.url).netloc.removeprefix('www.'))
    published = clean(tf.get('date') or ld.get('published_at') or meta(soup,'meta[property="article:published_time"]'))
    ok = bool(title and body)
    return {'article_id':article_id,'final_url':response.url,'title':title,'topic':topic,'topic_source':topic_source,'summary':summary,'summary_source':summary_source,'body':body,'published_at':published,'publisher':publisher,'http_status':response.status_code,'fetch_status':'success' if ok else 'incomplete','error':'' if ok else '제목 또는 본문 추출 실패','fetched_at_utc':datetime.now(timezone.utc).isoformat(),'ok':ok}

def read_cache(path: Path):
    try: return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
    except Exception: return None
def write_cache(path: Path, data: dict):
    temp = path.with_suffix('.tmp'); temp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8'); temp.replace(path)

targets = articles if CRAWL_LIMIT is None else articles.head(CRAWL_LIMIT)
stats = {'cached':0,'success':0,'failed_or_incomplete':0}
for row in tqdm(targets.itertuples(index=False), total=len(targets)):
    path = CACHE_DIR / f'{row.article_id}.json'; cached = read_cache(path)
    if cached and (cached.get('ok') or not RETRY_FAILED_CACHE): stats['cached'] += 1; continue
    try: data = extract_url(row.article_link,row.article_id)
    except Exception as exc:
        data = {'article_id':row.article_id,'final_url':'','title':'','topic':'','topic_source':'missing','summary':'','summary_source':'missing','body':'','published_at':'','publisher':'','http_status':getattr(getattr(exc,'response',None),'status_code',None),'fetch_status':'failed','error':f'{type(exc).__name__}: {exc}','fetched_at_utc':datetime.now(timezone.utc).isoformat(),'ok':False}
    stats['success' if data['ok'] else 'failed_or_incomplete'] += 1
    write_cache(path,data); time.sleep(REQUEST_DELAY_SECONDS)
stats

## 4. 카테고리별 CSV 저장 및 품질 확인

In [ ]:
records = [data for row in articles.itertuples(index=False) if (data := read_cache(CACHE_DIR / f'{row.article_id}.json'))]
extracted = pd.DataFrame(records) if records else pd.DataFrame(columns=['article_id'])
output = legacy.merge(extracted,on='article_id',how='left',validate='many_to_one')
output['fetch_status'] = output.get('fetch_status',pd.Series(index=output.index,dtype=str)).fillna('not_fetched')
columns = ['digest_date','legacy_category','article_link','article_id','title','topic','topic_source','summary','summary_source','body','published_at','publisher','final_url','http_status','fetch_status','error','fetched_at_utc','source_file']
for column in columns:
    if column not in output: output[column] = ''
output = output[columns].sort_values(['digest_date','legacy_category','article_id']).reset_index(drop=True)
saved_files = {}
for source_file, category_output in output.groupby('source_file', sort=True):
    output_path = OUTPUT_DIR / f'{Path(source_file).stem}_enriched.csv'
    category_output.to_csv(output_path,index=False,encoding='utf-8-sig')
    saved_files[source_file] = {'path': str(output_path), 'rows': len(category_output)}
display(output.head())
display(output['fetch_status'].value_counts(dropna=False).to_frame('rows'))
display(output['summary_source'].fillna('missing').value_counts().to_frame('rows'))
success = output['fetch_status'].eq('success')
sample_size = min(10,int(success.sum()))
if sample_size: display(output.loc[success,['publisher','title','topic','summary','summary_source','body']].sample(sample_size,random_state=42))
print('카테고리별 저장 완료')
display(pd.DataFrame(saved_files).T)